# 04c — Coastal distance and house type

Two more features (plan Part 2), plus a refit to see how much they add on
top of Model 2 (floor area + relative_size, HPI-adjusted; test R² 0.560,
MdAPE 18.0%).

**House type** is nearly free — we already have `built_form` attached to
every sale (notebook 04), and the EDA pass found it's one of the strongest
signals in the whole dataset (detached £294,995 vs terrace £95,000–£125,000
median — a much bigger spread than anything else we've measured). It just
hasn't been given to a model yet.

**Coastal distance** needs new data: OpenStreetMap's map of Sefton
(`merseyside-latest.osm.pbf`, 23.8 MB, downloaded with permission). We pull
out beach polygons and measure straight-line distance from each property
to the nearest one, in metres, using the British National Grid projection
(not raw lat/lon degrees — see the warning in Part B).


In [1]:
import sys
sys.path.insert(0, "..")
from src.config import DATA_RAW, DATA_INTERIM
import pandas as pd
import numpy as np
import geopandas as gpd
import statsmodels.api as sm

df = pd.read_parquet(DATA_INTERIM / "ppd_epc_joined_sefton_features.parquet")
print(f"{len(df):,} transactions with features from notebook 04")


61,344 transactions with features from notebook 04


## Part A — house type

`built_form` is a category (Detached, Semi-Detached, Mid-Terrace, ...), not
a number, so a straight-line model needs it turned into a set of yes/no
columns — e.g. `is_detached` (1 if detached, 0 otherwise) — one per
category. This is called **one-hot encoding**. We drop one category
(terraces, the most common) as the "baseline" that the others are compared
against, which is standard practice and avoids redundant columns.


In [2]:
# Collapse rare sub-categories (Enclosed Mid/End-Terrace) into their plain
# counterparts - EDA showed them behaving similarly and some have very few
# examples, which makes their own dummy coefficient unstable.
built_form_map = {
    "Enclosed Mid-Terrace": "Mid-Terrace",
    "Enclosed End-Terrace": "End-Terrace",
}
df["built_form_clean"] = df["built_form"].replace(built_form_map)

dummies = pd.get_dummies(df["built_form_clean"], prefix="type", drop_first=True, dtype=float)
print("Categories (one dropped as baseline):", dummies.columns.tolist())
df = pd.concat([df, dummies], axis=1)


Categories (one dropped as baseline): ['type_End-Terrace', 'type_Mid-Terrace', 'type_Not Recorded', 'type_Semi-Detached']


## Part B — coastal distance

**Why British National Grid, not raw lat/lon.** A degree of longitude
covers a different real-world distance than a degree of latitude (more so
the further from the equator), so measuring "distance" directly in lat/lon
numbers gives a distorted answer. EPSG:27700 (British National Grid) is a
flat-earth projection designed for exactly this — coordinates in it are
plain metres on a grid, the same system British maps use.

**Why beach polygons, not the general coastline.** Sefton's shoreline runs
from Crosby beach into Seaforth container terminal further south. Measuring
to the nearest bit of *any* coastline would score a house backing onto a
working port the same as one facing the beach. Beach polygons are tagged
in OpenStreetMap specifically as `natural=beach` — nobody tags a container
terminal that way, so this measure only ever finds genuine open beach.


In [3]:
osm_path = DATA_RAW / "osm" / "merseyside-latest.osm.pbf"

beaches = gpd.read_file(osm_path, layer="multipolygons", where="natural = 'beach'")
print(f"{len(beaches)} beach polygons found")

marine_lakes = gpd.read_file(osm_path, layer="multipolygons", where="natural = 'water'")
marine_lakes = marine_lakes[marine_lakes["name"].str.contains("Marine Lake", case=False, na=False)]
print(f"{len(marine_lakes)} Marine Lake polygons found")


72 beach polygons found


4 Marine Lake polygons found


In [4]:
BNG = "EPSG:27700"
beaches_bng = beaches.to_crs(BNG)
lakes_bng = marine_lakes.to_crs(BNG)

# One combined shape per feature type - much faster to measure distance to
# a single unified shape than to loop over 72 separate beach polygons.
beach_shape = beaches_bng.union_all()
lake_shape = lakes_bng.union_all()

properties = df.dropna(subset=["lat", "lon"]).copy()
points = gpd.GeoSeries(
    gpd.points_from_xy(properties["lon"], properties["lat"]), crs="EPSG:4326"
).to_crs(BNG)

properties["dist_to_beach_m"] = points.distance(beach_shape).values
properties["dist_to_marine_lake_m"] = points.distance(lake_shape).values

print(properties[["dist_to_beach_m", "dist_to_marine_lake_m"]].describe())


       dist_to_beach_m  dist_to_marine_lake_m
count     61310.000000           61310.000000
mean       3081.014261            3903.019049
std        2104.657078            2761.469396
min          42.709885              44.814826
25%        1651.344208            1832.563479
50%        2553.298208            2931.566211
75%        3739.382016            5918.191789
max       10508.958636           10487.768087


## Sanity checks

1. No property should show as essentially on top of the beach *and* be in
   Bootle/Seaforth (postcodes L20/L21, where the docks are) — that would
   mean the docks were accidentally being scored as beach.
2. Beach and Marine Lake distance were flagged in the plan as likely
   redundant (~200 m apart in reality) — check the correlation between them
   before deciding whether to keep both.


In [5]:
dock_area = properties[properties["postcode"].str.startswith(("L20", "L21"))]
print(f"Closest a Bootle/Seaforth (L20/L21) property gets to 'beach': "
      f"{dock_area['dist_to_beach_m'].min():.0f} m "
      f"(should be a real distance, not ~0)")

corr = properties[["dist_to_beach_m", "dist_to_marine_lake_m"]].corr().iloc[0, 1]
print(f"\nCorrelation between beach distance and Marine Lake distance: {corr:+.3f}")
print("Keeping both if clearly separate, dropping one if this is above ~0.9 as expected.")


Closest a Bootle/Seaforth (L20/L21) property gets to 'beach': 791 m (should be a real distance, not ~0)

Correlation between beach distance and Marine Lake distance: +0.557
Keeping both if clearly separate, dropping one if this is above ~0.9 as expected.


## Does coastal distance actually correlate with price?


In [6]:
have_price = properties.dropna(subset=["price_adjusted"])
log_price = np.log(have_price["price_adjusted"])
corr_beach = np.corrcoef(have_price["dist_to_beach_m"], log_price)[0, 1]
print(f"log(price) vs distance to beach: corr = {corr_beach:+.3f}  "
      f"(negative expected - further from the beach, cheaper)")


log(price) vs distance to beach: corr = -0.064  (negative expected - further from the beach, cheaper)


## Refit: Model 3 — floor area + relative_size + house type + coastal distance

Same temporal split as every prior check, so this stays comparable to
Model 1 (0.492) and Model 2 (0.560).


In [7]:
model_df = properties.dropna(subset=["total_floor_area", "relative_size", "price_adjusted", "dist_to_beach_m"]).copy()
model_df["log_price_adj"] = np.log(model_df["price_adjusted"])
model_df["log_area"] = np.log(model_df["total_floor_area"])
model_df["log_relsize"] = np.log(model_df["relative_size"])
model_df["dist_to_beach_km"] = model_df["dist_to_beach_m"] / 1000  # easier-to-read coefficient

built_form_cols = dummies.columns.tolist()
feature_cols = ["log_area", "log_relsize", "dist_to_beach_km"] + built_form_cols

split_date = pd.Timestamp("2024-07-01")
train = model_df[model_df["date_of_transfer"] < split_date]
test = model_df[model_df["date_of_transfer"] >= split_date]
print(f"Train: {len(train):,} | Test: {len(test):,}")

X_train = sm.add_constant(train[feature_cols].astype(float))
model3 = sm.OLS(train["log_price_adj"], X_train).fit()

X_test = sm.add_constant(test[feature_cols].astype(float))
pred_price = np.exp(model3.predict(X_test))
actual = test["price_adjusted"]
ape = (pred_price - actual).abs() / actual
mdape = ape.median()
ppe10 = (ape <= 0.10).mean()
ss_res = ((test["log_price_adj"] - np.log(pred_price)) ** 2).sum()
ss_tot = ((test["log_price_adj"] - test["log_price_adj"].mean()) ** 2).sum()
test_r2 = 1 - ss_res / ss_tot

print(f"\nModel 3: floor area + relative_size + house type + coastal distance")
print(f"Train R²:   {model3.rsquared:.3f}")
print(f"Test R²:    {test_r2:.3f}")
print(f"Test MdAPE: {mdape:.1%}")
print(f"Test PPE10: {ppe10:.1%}")


Train: 46,789 | Test: 6,139



Model 3: floor area + relative_size + house type + coastal distance
Train R²:   0.656
Test R²:    0.668
Test MdAPE: 15.1%
Test PPE10: 34.7%


## What's driving the price, in plain terms

OLS coefficients on a log target translate to "roughly this % change in
price per unit change in the feature" — read the house-type coefficients
against the Mid-Terrace baseline they were built relative to.


In [8]:
print(model3.summary())


                            OLS Regression Results                            
Dep. Variable:          log_price_adj   R-squared:                       0.656
Model:                            OLS   Adj. R-squared:                  0.656
Method:                 Least Squares   F-statistic:                 1.274e+04
Date:                Sat, 08 Aug 2026   Prob (F-statistic):               0.00
Time:                        10:01:05   Log-Likelihood:                -12108.
No. Observations:               46789   AIC:                         2.423e+04
Df Residuals:                   46781   BIC:                         2.430e+04
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                  8.1189      0

## Save


In [9]:
properties.to_parquet(DATA_INTERIM / "ppd_epc_joined_sefton_features_v2.parquet", index=False)
print("Saved.")


Saved.
